# Topic Modelling

### 1 - Setup

In [ ]:
from foundry.transforms import Dataset
import pandas as pd
import numpy as np
from sklearn.cluster import AffinityPropagation
import json
import re
import ast
from string import Template

from language_model_service_api.languagemodelservice_api_embeddings_v3 import GenericEmbeddingsRequest
from language_model_service_api.languagemodelservice_api_completion_v3 import GptChatCompletionRequest
from language_model_service_api.languagemodelservice_api import ChatMessage, ChatMessageRole
from palantir_models.models import GenericEmbeddingModel
from palantir_models.models import OpenAiGptChatLanguageModel

from config import config, datasets
from src.topic_modelling import topic_modelling

class PromptTemplate(Template):
    delimiter = "$"

In [ ]:
labels_column = config["labels_column"]
topics_property = config["topics_property"]

response_summaries = topic_modelling.get_topic_list_from_datasets(
    datasets, 
    labels_column, 
    topics_property, 
    config
)

### 2 - LLM for Topic Modelling

In [ ]:
extraction_prompt_new_topics = PromptTemplate("""
You are categorizing "Ideas for Change" from survey responses into topics. You are an expert at topic modelling.

- The "Current Topics" are: $topics_
- The "Ideas for Change" are: $ideas_for_change

Instructions:
1. Assign the "Ideas for Change" to one or more topics from the "Current Topics".
2. All assigned topics must be in lower case, with the exact same spelling as the topic found in "Current Topics".
3. If the appropriate topic does not exist in "Current Topics", generate a new topic and include it in your response. Give your new topic a descriptive name, in lowercase. Creating a new topic is optional, do not create a new topic if the "Ideas for Change" can be well described by existing topics.
4. If the "Ideas for Change" applies to multiple topics, you may assign it to up to five topics, some of which may be new.
5. Do not assign more than five topics.
6. If there are no ideas for change, assign a "none" topic. 

Example:
If the "Current Topics" are ["improved communication", "employee wellbeing", "none"], the "Ideas for Change" are "Install solar panels on the roof and improve communication between staff members." your response might be:
[sustainability", "improved communication"]
The reasoning for this is that both "improved communication" and "sustainability" describe these ideas for change. "improved communication" already exists in "Current Topics", but "sustainability" is a new topic.

Your response must strictly be a Python list containing the relevant topics, such as:
["topic a", "topic b"]

Do not include any additional text in your response, only a python list.
Do not label a topic as a "new topic" - instead give it a descriptive name.
""")

extraction_prompt_existing_topics = PromptTemplate("""
You are categorizing "Ideas for Change" from survey responses into topics. You are an expert at topic modelling.

- The "Current Topics" are: $topics_
- The "Ideas for Change" are: $ideas_for_change

Instructions:
1. Assign the "Ideas for Change" to one or more topics from the "Current Topics".
2. If the appropriate topic does not exist in "Current Topics", select the most similar topic from the list.
3. All assigned topics must be in lower case, with the exact same spelling as the topic found in "Current Topics".
4. If the "Ideas for Change" applies to multiple topics, you may assign it to up to five topics.
5. Do not assign more than five topics.
6. If there are no ideas for change, assign a "none" topic. 

Example:
If the "Current Topics" are ["improved communication", "employee wellbeing", "none"], the "Ideas for Change" are "Install solar panels on the roof and improve communication between staff members." your response might be:
[sustainability", "improved communication"]
The reasoning for this is that both "improved communication" and "sustainability" describe these ideas for change. "improved communication" already exists in "Current Topics", but "sustainability" is a new topic.

Your response must strictly be a Python list containing the relevant topics, such as:
["topic a", "topic b"]

Do not include any additional text in your response, only a python list.
Do not label a topic as a "new topic" - instead give it a descriptive name.
""")

cleaning_prompt = PromptTemplate("""Clean a string containing a list of topics to be a flat python list.
Your response must strictly be a flat python list containing all of the topics, with no additional information.

Example 1:
If the input is: [["topic 1"], ["topic 2"]]
Your output must be: ["topic 1", "topic 2"]

Example 2:
If the input is: [["topic 1", 'topic 2'],"topic 3"]
Your output must be: ["topic 1", "topic 2", "topic 3"]

Example 3:
If the input is: [["topic 1", "topic 2", "topic 3", "topic 4"], "topic 5", "topic A", ]]
Your output must be: ["topic 1", "topic 2", "topic 3", "topic 4", "topic 5", "topic A"]

Example 4:
If the input is: ["communication", "improve leadership"]
Your output must be: ["communication", "improve leadership"]

Example 5: Certainly, here is the cleaned list you requested: ["increase the amount of staff training", "more al"]
Your output must be: ["increase the amount of staff training", "more al"]

Example 6: ["better education", "more oppertunities for progression"] - if you need any more help, please feel free to ask.
Your output must be: ["better education", "more oppertunities for progression"]

Rules:
- Do not include any commentary or markdown tags in your response.
- Do not include a variable assignment.
- All topic spelling must remain the same, do not change or correct any spelling.
- Your response should be valid python list syntax only - that is, starting with an opening square bracket, then a comma-separated list of the topics, each in quotation marks, and a closing square bracket.

The string you must simplify is: $response"""
)

In [ ]:
extraction_prompt = extraction_prompt_new_topics if config["add_new_topics"] == True else extraction_prompt_existing_topics

current_topics = config["starting_topics"]
print(f"Starting Topics: {current_topics}")

all_responses = []
failed_responses = []

model = OpenAiGptChatLanguageModel.get("GPT_4o")

for i in range(len(response_summaries)):
    
    if i%100==0 and i!=0:
        print(f"Completed {i}/{len(response_summaries)}")
        print(f"Current Topics: {current_topics}")
    
    formatted_prompt = extraction_prompt.safe_substitute(
            topics_ = current_topics,
            ideas_for_change = response_summaries[i][labels_column][topics_property])
    response = model.create_chat_completion(GptChatCompletionRequest([ChatMessage(ChatMessageRole.USER, formatted_prompt)],temperature = config["temperature"]))
    raw_content = response.choices[0].message.content  # Extract the raw content
    
    try:
        topic_array = ast.literal_eval(raw_content)
    except:
        formatted_cleaning_prompt = cleaning_prompt.safe_substitute(response=raw_content)
        response = model.create_chat_completion(GptChatCompletionRequest([ChatMessage(ChatMessageRole.USER, formatted_cleaning_prompt)],temperature = config["temperature"]))
        raw_content = response.choices[0].message.content  # Extract the raw content        
        try:
            topic_array = ast.literal_eval(raw_content)
        except:
            print("Failed at",i)
            print("with", raw_content)
            all_responses.append("ERROR: ")
            failed_responses.append([i, raw_content]) 
            continue
        
    if topic_modelling.contains_nests(topic_array):
        topic_array = topic_modelling.flatten_list(topic_array)
    topic_array = {"tm_id": response_summaries[i]["tm_id"], "topics": topic_array}
    all_responses, topic_array, current_topics = topic_modelling.add_topics(all_responses, topic_array, current_topics, config["add_new_topics"])
        

### 3 - Error Handling

In [ ]:
# Let's look at the current topics

print(current_topics)

In [ ]:
# Manual Cleaning Steps may be needed

print(failed_responses)

In [ ]:
# If there are rows which you need to manually clean, fill the follopwing section.

# This will contain the indexes and correctly cleaned lists
incorrect_cleaned_lists = {} 
    #"1": [cleaned_list],
    #"2": [cleaned_list]
#}

In [ ]:
cleaned_ideas = []
cleaned_responses = []

for idea in response_summaries:
    if isinstance(idea[labels_column][topics_property], str):
        cleaned_ideas.append(idea[labels_column][topics_property])
    else:
        cleaned_ideas.append("None")
    

for i, response in enumerate(all_responses):
    if isinstance(response["topics"], list):
        cleaned_responses.append(response["topics"])
    elif i in list(incorrect_cleaned_lists.keys()):
        cleaned_responses.append(incorrect_cleaned_lists[f"{i}"])
    else:
        cleaned_responses.append(["ERROR"])

### 4 - Saving and Displaying

In [ ]:
print("The number of topics are:", len(current_topics))

df = pd.DataFrame(
    {
        "tm_id": [idea["tm_id"] for idea in response_summaries],
        topics_property: cleaned_ideas, 
        "Topics": cleaned_responses
    }
)

print("Modelling failed at:", [f[0] for f in failed_responses])

if config["test_mode"]:
    display(df)
else:
    llm_topic_modeling = Dataset.get("llm_topic_modelling")
    llm_topic_modeling.write_table(df)

In [ ]:
df_topic_counts = pd.DataFrame({
    "Topic": [response["topics"] for response in all_responses],
    "Cluster": [idea[labels_column][topics_property] for idea in response_summaries]
})
df_topic_counts = (df_topic_counts
    .explode("Topic")
    .query(f"Topic in {current_topics}")
    .groupby("Topic")
    .agg(
        Count=('Topic', 'count'),
        Cluster=('Cluster', lambda x: list(x)),
    )
    .sort_values('Count', ascending=False)
    .reset_index()
)

if config["test_mode"]:
    display(df_topic_counts)
else:
    llm_topic_counts = Dataset.get("llm_topic_counts")
    llm_topic_counts.write_table(pd_topic_df)